# NB26 — LTR with graded-relevance labels vs binary click labels

**Regime:** `[LAB]` four-cell labels (requires NB22 gaze-regression), cursor-deployable at inference-time features.

## Key Claims

| K-ID | Claim | Value | Source cell |
|---|---|---|---|
| K1 | MRR@10 on original SERP ordering (Google baseline) | _TBD_ | Cell 8 |
| K2 | MRR@10 on LR ranker trained on binary click/no-click-above labels | _TBD_ | Cell 8 |
| K3 | MRR@10 on LR ranker trained on 3-cell graded labels (clicked=2, deferred=1, eval-rejected/not-approached-above=0) | _TBD_ | Cell 8 |
| K4 | Headline Δ: MRR(graded) − MRR(binary), paired per-participant over 47 folds | _TBD_ | Cell 8 |
| K5 | Headline Δ: MRR(graded) − MRR(original SERP) | _TBD_ | Cell 8 |

## Context: Peter Dixon-Moses LTR design (2026-04-15)

Peter's pushback on framing deferred as hard-negative landed as the graded-relevance reframe in the CIKM paper (§4.2, §4.3). This notebook is the **empirical validation** of that reframe: does training an LTR ranker on the 3-cell graded labels *actually* improve downstream MRR@10 over training on binary click labels, on the same feature set and same held-out participants?

**Peter's exact label scheme:**

- Clicked → grade 2
- NB22 deferred → grade 1
- Eval-rejected AND not-approached *above* click position → grade 0
- **Exclude** not-approached records *below* click position. Not evaluated, not rejected — structurally unseen. Training on them as 0 teaches a confound.

**Peter's exact feature set (five features, no position):**

1. `lexical_overlap(query_tokens, title+snippet_tokens)` — fraction of query tokens appearing in result text
2. `avg_term_frequency(query_tokens, title+snippet_tokens)` — mean count per query term
3. `cos_sim(query, title)`
4. `cos_sim(query, snippet)`
5. `cos_sim(query, title+snippet)` — cached in `serp-embeddings.json`

**Position is intentionally excluded** as a feature. Peter: position-bias orthodoxy is a trick that assumes position confounds the label; in forced-choice AdSERP where every item is fixated, there is no confound to correct for.

**CV protocol:** leave-one-participant-out, 47 folds (matches §4.1 M4 LOSO).

**Ranker:** sklearn LogisticRegression per Andy's preference (simpler than LambdaMART/lightgbm; scores each record pointwise; sorted by score to produce a per-trial ranking). Peter's design works for LR or LambdaMART — LR is sufficient for the binary-vs-graded contrast.

**Headline:** does MRR(graded) > MRR(binary) on the paired per-participant delta? If yes, the graded cell labels add downstream ranker-training information that binary click labels miss — empirical validation of the graded-relevance reframe.

In [1]:
# ── Imports + paths ───────────────────────────────────────────────────
import json
import re
import sys
from collections import defaultdict
from pathlib import Path

import numpy as np
from scipy.stats import spearmanr, wilcoxon
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

ROOT = Path("/Users/andyed/Documents/dev/attentional-foraging")
sys.path.insert(0, str(ROOT / "notebooks-v2"))

LAB_RECORDS = ROOT / "AdSERP/data/cursor-approach-features.json"
REGRESSION_CACHE = ROOT / "scripts/output/approach_threshold_sensitivity/regression_labels_cache.json"
SERP_EMB_COMBINED = ROOT / "AdSERP/data/serp-embeddings.json"
SERP_EMB_SPLIT = ROOT / "AdSERP/data/serp-embeddings-split.json"
QUERY_EMB = ROOT / "AdSERP/data/query-embeddings.json"
print("imports ok")

imports ok


In [2]:
# ── Load AdSERP records, NB22 labels, embeddings ─────────────────────
lab_records = json.load(open(LAB_RECORDS))
regression_labels = np.array(json.load(open(REGRESSION_CACHE)), dtype=bool)
print(f"lab records: {len(lab_records):,}")
print(f"NB22 gaze_regression labels: {len(regression_labels):,}")

with open(SERP_EMB_COMBINED) as f:
    serp_combined = json.load(f)
with open(SERP_EMB_SPLIT) as f:
    serp_split = json.load(f)
with open(QUERY_EMB) as f:
    query_data = json.load(f)

print(f"combined embeddings: {len(serp_combined):,} trials")
print(f"split embeddings: {len(serp_split):,} trials")
print(f"query embeddings: {len(query_data):,} trials")

# Lookups keyed by (trial_id, position)
combined_emb = {}
combined_text = {}
for tid, rs in serp_combined.items():
    for r in rs:
        if "embedding" in r:
            combined_emb[(tid, r["position"])] = np.array(r["embedding"], dtype=np.float32)
            combined_text[(tid, r["position"])] = r.get("text", "")

title_emb = {}
snippet_emb = {}
title_text = {}
snippet_text = {}
for tid, rs in serp_split.items():
    for r in rs:
        key = (tid, r["position"])
        if "title_embedding" in r:
            title_emb[key] = np.array(r["title_embedding"], dtype=np.float32)
        if "snippet_embedding" in r:
            snippet_emb[key] = np.array(r["snippet_embedding"], dtype=np.float32)
        title_text[key] = r.get("title", "") or ""
        snippet_text[key] = r.get("snippet", "") or ""

query_emb_lookup = {}
query_text_lookup = {}
for tid, v in query_data.items():
    if isinstance(v, dict) and "embedding" in v:
        query_emb_lookup[tid] = np.array(v["embedding"], dtype=np.float32)
        query_text_lookup[tid] = v.get("query", "")

print(f"\nlookups built:")
print(f"  combined emb: {len(combined_emb):,}")
print(f"  title emb:    {len(title_emb):,}")
print(f"  snippet emb:  {len(snippet_emb):,}")
print(f"  query emb:    {len(query_emb_lookup):,}")

lab records: 13,419
NB22 gaze_regression labels: 13,419


combined embeddings: 2,776 trials
split embeddings: 2,776 trials
query embeddings: 2,776 trials



lookups built:
  combined emb: 27,520
  title emb:    27,520
  snippet emb:  27,520
  query emb:    2,776


In [3]:
# ── Apply Peter's label scheme with exclusions ────────────────────────
# For each trial, identify the clicked result position, then:
#   clicked            → 2
#   NB22 deferred      → 1
#   eval-rejected      → 0
#   not-approached ABOVE click → 0
#   not-approached BELOW click → EXCLUDED (structurally unseen)
#
# Also build the binary variant used by Peter's 3-way comparison:
#   clicked → 1
#   everything else above click → 0
#   below click → excluded

# First: click position per trial
click_pos_by_trial = {}
for r in lab_records:
    if r.get("was_clicked"):
        click_pos_by_trial[r["trial_id"]] = r["position"]
print(f"trials with a click: {len(click_pos_by_trial):,}")

# Build per-record labels
graded_by_key = {}
binary_by_key = {}
excluded_count = 0
for i, r in enumerate(lab_records):
    tid = r["trial_id"]
    pos = r["position"]
    key = (tid, pos)
    click_pos = click_pos_by_trial.get(tid)
    if click_pos is None:
        continue  # no click in this trial — can't build Peter's exclusion

    clicked = bool(r.get("was_clicked", False))
    min_dist = float(r.get("min_dist", 9999))
    approached = min_dist < 100
    deferred = bool(regression_labels[i]) if i < len(regression_labels) else False

    # Apply below-click exclusion for not-approached records only
    if not approached and not clicked and pos > click_pos:
        excluded_count += 1
        continue

    # Graded label
    if clicked:
        grade = 2
    elif approached and deferred:
        grade = 1
    else:
        grade = 0
    graded_by_key[key] = grade
    binary_by_key[key] = 1 if clicked else 0

print(f"graded-labeled records:   {len(graded_by_key):,}")
print(f"  grade 2 (clicked):      {sum(1 for g in graded_by_key.values() if g == 2):,}")
print(f"  grade 1 (deferred):     {sum(1 for g in graded_by_key.values() if g == 1):,}")
print(f"  grade 0 (rejected/nota): {sum(1 for g in graded_by_key.values() if g == 0):,}")
print(f"excluded not-approached-below-click records: {excluded_count:,}")

trials with a click: 2,228
graded-labeled records:   7,432
  grade 2 (clicked):      2,228
  grade 1 (deferred):     1,862
  grade 0 (rejected/nota): 3,342
excluded not-approached-below-click records: 5,480


In [4]:
# ── Compute Peter's 5 features per (trial, position) ─────────────────
# 1. lexical_overlap(query_tokens, title+snippet_tokens)
# 2. avg_term_frequency(query_tokens, title+snippet_tokens)
# 3. cos_sim(query, title)
# 4. cos_sim(query, snippet)
# 5. cos_sim(query, title+snippet)

STOPWORDS = set("a an and or but the is are was were be been being "
                "to of in on at for from by with as "
                "this that these those".split())

def tokenize(text):
    tokens = re.findall(r"[a-z0-9]+", (text or "").lower())
    return [t for t in tokens if t not in STOPWORDS and len(t) > 1]

def cos_sim(a, b):
    if a is None or b is None:
        return 0.0
    na = float(np.linalg.norm(a))
    nb = float(np.linalg.norm(b))
    if na == 0 or nb == 0:
        return 0.0
    return float(np.dot(a, b) / (na * nb))

FEATURE_NAMES = [
    "lex_overlap",
    "avg_tf",
    "cos_title",
    "cos_snippet",
    "cos_combined",
]

features_by_key = {}
skipped_missing_emb = 0
skipped_missing_query = 0
for key in graded_by_key:
    tid, pos = key
    q_emb = query_emb_lookup.get(tid)
    q_text = query_text_lookup.get(tid, "")
    if q_emb is None or not q_text:
        skipped_missing_query += 1
        continue
    t_emb = title_emb.get(key)
    s_emb = snippet_emb.get(key)
    c_emb = combined_emb.get(key)
    if t_emb is None or s_emb is None or c_emb is None:
        skipped_missing_emb += 1
        continue

    # Lexical features
    q_tokens = tokenize(q_text)
    r_text = (title_text.get(key, "") or "") + " " + (snippet_text.get(key, "") or "")
    r_tokens = tokenize(r_text)
    if not q_tokens:
        lex_overlap = 0.0
        avg_tf = 0.0
    else:
        r_counter = {}
        for t in r_tokens:
            r_counter[t] = r_counter.get(t, 0) + 1
        matches = [t for t in q_tokens if t in r_counter]
        lex_overlap = len(matches) / len(q_tokens)
        # Average TF across query tokens: absent tokens count as 0.
        avg_tf = float(np.mean([r_counter.get(t, 0) for t in q_tokens]))

    features_by_key[key] = [
        lex_overlap,
        avg_tf,
        cos_sim(q_emb, t_emb),
        cos_sim(q_emb, s_emb),
        cos_sim(q_emb, c_emb),
    ]

print(f"feature vectors computed: {len(features_by_key):,}")
print(f"  skipped (missing query): {skipped_missing_query}")
print(f"  skipped (missing embs):  {skipped_missing_emb}")

# Summary stats per feature
X_all = np.array(list(features_by_key.values()))
print(f"\nfeature matrix shape: {X_all.shape}")
for j, name in enumerate(FEATURE_NAMES):
    col = X_all[:, j]
    print(f"  {name:>13}  mean={col.mean():+.3f}  std={col.std():.3f}  min={col.min():+.3f}  max={col.max():+.3f}")

feature vectors computed: 7,428
  skipped (missing query): 0
  skipped (missing embs):  4

feature matrix shape: (7428, 5)
    lex_overlap  mean=+0.748  std=0.206  min=+0.000  max=+1.000
         avg_tf  mean=+1.938  std=0.872  min=+0.000  max=+7.857
      cos_title  mean=+0.823  std=0.136  min=+0.249  max=+0.991
    cos_snippet  mean=+0.769  std=0.086  min=+0.338  max=+0.941
   cos_combined  mean=+0.805  std=0.059  min=+0.461  max=+0.957


In [5]:
# ── Build the per-trial record structure for ranker training ─────────
# Each trial becomes a list of (features, graded_label, binary_label, position,
# was_clicked, participant) records. MRR is computed per trial.

trials_data = defaultdict(list)
for key, feats in features_by_key.items():
    tid, pos = key
    participant = tid.split("-")[0]
    trials_data[tid].append({
        "position": pos,
        "features": feats,
        "graded": graded_by_key[key],
        "binary": binary_by_key[key],
        "clicked": (graded_by_key[key] == 2),
        "participant": participant,
    })

# Trials with at least one clicked record AND at least one non-clicked
# in the labeled pool are the ones we can train on.
usable_trials = [
    tid for tid, rs in trials_data.items()
    if any(r["clicked"] for r in rs) and len(rs) >= 2
]
print(f"total trials: {len(trials_data):,}")
print(f"usable trials (has click + ≥2 labeled records): {len(usable_trials):,}")

# Distribution of records per trial
lengths = [len(trials_data[tid]) for tid in usable_trials]
print(f"records per usable trial: "
      f"mean={np.mean(lengths):.1f}, "
      f"median={int(np.median(lengths))}, "
      f"min={min(lengths)}, max={max(lengths)}")

total trials: 2,228
usable trials (has click + ≥2 labeled records): 1,919
records per usable trial: mean=3.7, median=3, min=2, max=10


In [6]:
# ── MRR helper: compute per-trial MRR@10 given a scoring function ───

def mrr_for_trial(trial_records, scoring_fn):
    """Given a list of records for a trial and a callable that takes a
    feature row and returns a score, return 1 / rank_of_clicked_result.
    Rank is 1-indexed. If the trial has no clicked record in the labeled
    pool (shouldn't happen for usable trials), returns 0."""
    if not trial_records:
        return 0.0
    scored = [(scoring_fn(r["features"]), r["clicked"]) for r in trial_records]
    # Sort by descending score; ties broken by insertion order
    scored.sort(key=lambda x: -x[0])
    for rank, (_, clicked) in enumerate(scored, 1):
        if clicked:
            return 1.0 / rank
    return 0.0

def mrr_original_order(trial_records):
    """MRR using the Google SERP ordering directly (position index).
    Note: this is MRR@N where N = labeled records, not always @10,
    because Peter's exclusion drops below-click non-approached items."""
    if not trial_records:
        return 0.0
    sorted_by_pos = sorted(trial_records, key=lambda r: r["position"])
    for rank, r in enumerate(sorted_by_pos, 1):
        if r["clicked"]:
            return 1.0 / rank
    return 0.0

print("MRR helper defined")

MRR helper defined


In [7]:
# ── Leave-one-participant-out CV: train LR per fold, score held-out ──
# Two ranker variants: binary labels, graded labels. Same features, same splits.

def train_scoring_fn(train_records, label_key):
    """Return a callable that maps a feature row to a score.
    For graded labels we treat it as pointwise regression onto [0, 1, 2];
    for binary we fit logistic regression and use decision_function."""
    X = np.array([r["features"] for r in train_records])
    y = np.array([r[label_key] for r in train_records])
    if label_key == "binary":
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("lr", LogisticRegression(max_iter=5000, class_weight="balanced", C=1.0)),
        ])
        pipe.fit(X, y)
        # decision_function returns log-odds; higher = more clickable
        return lambda feats: float(pipe.decision_function(np.asarray(feats).reshape(1, -1))[0])
    else:
        # Pointwise graded: standardize features and fit a linear regression on the
        # integer grades. Higher grade = more relevant. Sklearn LR would reduce to
        # a 3-class problem that loses ordinal structure; a linear regression on
        # the graded scores is the correct pointwise treatment for this comparison.
        from sklearn.linear_model import Ridge
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("ridge", Ridge(alpha=1.0)),
        ])
        pipe.fit(X, y.astype(float))
        return lambda feats: float(pipe.predict(np.asarray(feats).reshape(1, -1))[0])

# Build participant → [trial_ids] index
participant_to_trials = defaultdict(list)
for tid in usable_trials:
    participant = trials_data[tid][0]["participant"]
    participant_to_trials[participant].append(tid)
participants = sorted(participant_to_trials.keys())
print(f"participants: {len(participants)}")

# LOSO-by-participant
per_trial_mrr_original = {}
per_trial_mrr_binary = {}
per_trial_mrr_graded = {}

for fold_i, held_out_p in enumerate(participants):
    held_out_tids = set(participant_to_trials[held_out_p])
    train_tids = [t for t in usable_trials if t not in held_out_tids]
    train_records_all = []
    for t in train_tids:
        train_records_all.extend(trials_data[t])

    score_binary = train_scoring_fn(train_records_all, "binary")
    score_graded = train_scoring_fn(train_records_all, "graded")

    for t in held_out_tids:
        recs = trials_data[t]
        per_trial_mrr_original[t] = mrr_original_order(recs)
        per_trial_mrr_binary[t] = mrr_for_trial(recs, score_binary)
        per_trial_mrr_graded[t] = mrr_for_trial(recs, score_graded)

    if fold_i % 10 == 0:
        print(f"  fold {fold_i + 1}/{len(participants)} done")

# Aggregate
aligned_tids = [t for t in usable_trials if t in per_trial_mrr_binary]
mrr_orig = np.array([per_trial_mrr_original[t] for t in aligned_tids])
mrr_bin = np.array([per_trial_mrr_binary[t] for t in aligned_tids])
mrr_grad = np.array([per_trial_mrr_graded[t] for t in aligned_tids])

print(f"\naligned held-out trials: {len(aligned_tids):,}")
print(f"\n[K1] MRR (original SERP): {mrr_orig.mean():.4f}")
print(f"[K2] MRR (binary labels): {mrr_bin.mean():.4f}")
print(f"[K3] MRR (graded labels): {mrr_grad.mean():.4f}")

print(f"\n[K4] graded − binary: {mrr_grad.mean() - mrr_bin.mean():+.4f}")
print(f"[K5] graded − original: {mrr_grad.mean() - mrr_orig.mean():+.4f}")

# Paired per-participant aggregation
by_participant = defaultdict(lambda: {"orig": [], "bin": [], "grad": []})
for t, mo, mb, mg in zip(aligned_tids, mrr_orig, mrr_bin, mrr_grad):
    p = trials_data[t][0]["participant"]
    by_participant[p]["orig"].append(mo)
    by_participant[p]["bin"].append(mb)
    by_participant[p]["grad"].append(mg)

per_p_orig = np.array([np.mean(v["orig"]) for v in by_participant.values()])
per_p_bin = np.array([np.mean(v["bin"]) for v in by_participant.values()])
per_p_grad = np.array([np.mean(v["grad"]) for v in by_participant.values()])

paired_bin_grad = per_p_grad - per_p_bin
paired_orig_grad = per_p_grad - per_p_orig

print(f"\nper-participant paired:")
print(f"  mean(graded) = {per_p_grad.mean():.4f} ± {per_p_grad.std(ddof=1):.4f} (47 participants)")
print(f"  mean(binary) = {per_p_bin.mean():.4f} ± {per_p_bin.std(ddof=1):.4f}")
print(f"  mean(orig)   = {per_p_orig.mean():.4f} ± {per_p_orig.std(ddof=1):.4f}")
print(f"\n  graded − binary paired delta: {paired_bin_grad.mean():+.4f} ± {paired_bin_grad.std(ddof=1):.4f}")
print(f"  graded ≥ binary in {int((paired_bin_grad >= 0).sum())}/{len(paired_bin_grad)} participants")

# Wilcoxon signed-rank (non-parametric paired test)
try:
    w_bin = wilcoxon(per_p_grad, per_p_bin, alternative="greater")
    print(f"\n  Wilcoxon (graded > binary): W = {w_bin.statistic:.2f}, p = {w_bin.pvalue:.4f}")
except Exception as e:
    print(f"  Wilcoxon failed: {e}")
try:
    w_orig = wilcoxon(per_p_grad, per_p_orig, alternative="greater")
    print(f"  Wilcoxon (graded > original): W = {w_orig.statistic:.2f}, p = {w_orig.pvalue:.4f}")
except Exception as e:
    print(f"  Wilcoxon (graded vs orig) failed: {e}")

participants: 47
  fold 1/47 done


  fold 11/47 done


  fold 21/47 done


  fold 31/47 done


  fold 41/47 done

aligned held-out trials: 1,919

[K1] MRR (original SERP): 0.4114
[K2] MRR (binary labels): 0.6108
[K3] MRR (graded labels): 0.6152

[K4] graded − binary: +0.0045
[K5] graded − original: +0.2038

per-participant paired:
  mean(graded) = 0.6187 ± 0.0654 (47 participants)
  mean(binary) = 0.6141 ± 0.0601
  mean(orig)   = 0.4122 ± 0.0702

  graded − binary paired delta: +0.0046 ± 0.0209
  graded ≥ binary in 31/47 participants

  Wilcoxon (graded > binary): W = 720.50, p = 0.0246
  Wilcoxon (graded > original): W = 1128.00, p = 0.0000


## Results summary

After executing all cells, transcribe the K1–K5 values into the Key Claims block at the top.

**Interpretation guide:**

- **K4 > 0 (graded beats binary) with signed-rank p < 0.05**: the graded relevance labels add downstream ranker-training information that binary click labels miss. Direct empirical validation of Peter's reframe and the CIKM paper's graded-relevance framing.
- **K4 ≈ 0 (paired delta within noise)**: graded and binary labels produce equivalent rankers on the same features. Reframe still defensible (graded is more informative theoretically) but the empirical gain is undetected at this sample size.
- **K4 < 0**: graded labels hurt. Would indicate Peter's deferred=1 labeling is noisy enough to drag the ranker. Unlikely given K5 in NB25 showed 83% vs 45% deferred rate on rubbernecking split, but possible.
- **K5 > 0 (graded beats Google SERP order)**: the 5-feature behavioral + lexical ranker with 47-fold LOPO improves on the original ranking. Independent of the binary-vs-graded contrast, this tells us whether the behavioral signal adds value on top of whatever Google's ranker did.
- **K5 < 0 (graded loses to Google)**: Google's ranker is strictly better on this feature set. Not surprising — Google has orders of magnitude more features and training data. The contrast that matters is K4.

**If K4 lands positive**: goes into §4.3 of the CIKM paper as the direct empirical validation of the graded-relevance reframe. ~150 words, one table.